### 1. 정수 인코딩 (Integer Encoding)


In [1]:
from collections import Counter

# 1. 텍스트 데이터 준비
text = "사과 바나나 사과 딸기 사과 바나나"

# 2. 토큰화 (공백 기준)
tokens = text.split()

# 3. 빈도수 계산
vocab = Counter(tokens)
print("단어별 빈도수:", vocab)

# 4. 빈도수가 높은 순서대로 정렬
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1], reverse=True)
print()
print(f"sorted_vocab : {sorted_vocab}")

# 5. 고유 정수 부여 (1부터 시작)
word_to_index = {}
for i, (word, frequency) in enumerate(sorted_vocab, start=1):
    word_to_index[word] = i

print()
print("단어별 부여된 정수:", word_to_index)

# 6. 정수 인코딩 실행
encoded_text = [word_to_index[word] for word in tokens]
print()
print("원본 텍스트:", tokens)
print("정수 인코딩:", encoded_text)

단어별 빈도수: Counter({'사과': 3, '바나나': 2, '딸기': 1})

sorted_vocab : [('사과', 3), ('바나나', 2), ('딸기', 1)]

단어별 부여된 정수: {'사과': 1, '바나나': 2, '딸기': 3}

원본 텍스트: ['사과', '바나나', '사과', '딸기', '사과', '바나나']
정수 인코딩: [1, 2, 1, 3, 1, 2]


In [2]:
import torch
from collections import Counter

# 1. 텍스트 데이터 준비
sentences = ["나는 학교에 간다", "나는 집에도 가고 학교에도 간다", "학교는 즐겁다"]

# 2. 토큰화 (공백 기준)
tokenized_sentences = [sent.split() for sent in sentences]

# 3. 단어 빈도수 계산
tokens_all = [token for sent in tokenized_sentences for token in sent]
vocab_counts = Counter(tokens_all)

# 4. 단어 집합(Vocabulary) 생성 (스페셜 토큰 포함)
# <PAD>: 길이를 맞추기 위한 패딩 (0)
# <UNK>: 단어장에 없는 단어 (1)
word_to_idx = {"<PAD>": 0, "<UNK>": 1}

# 빈도순 정렬 후 인덱스 부여
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print("단어 집합(Vocab):", word_to_idx)


# 5. 정수 인코딩 (텍스트 -> 정수 리스트)
def encode(sentence_tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in sentence_tokens]


encoded_list = [encode(sent, word_to_idx) for sent in tokenized_sentences]

print()
print("encoded_list : ", encoded_list)
# 6. PyTorch Tensor로 변환
# (실제 입력 시에는 문장 길이를 맞추는 Padding 작업이 함께 들어갑니다)
tensor_data = [torch.tensor(seq, dtype=torch.long) for seq in encoded_list]

print()
print("tensor_data : ", tensor_data)

print()
print("--- 결과 ---")
for sent, tensor in zip(sentences, tensor_data):
    print(f"원문: {sent}")
    print(f"Tensor: {tensor}\n")

단어 집합(Vocab): {'<PAD>': 0, '<UNK>': 1, '나는': 2, '간다': 3, '학교에': 4, '집에도': 5, '가고': 6, '학교에도': 7, '학교는': 8, '즐겁다': 9}

encoded_list :  [[2, 4, 3], [2, 5, 6, 7, 3], [8, 9]]

tensor_data :  [tensor([2, 4, 3]), tensor([2, 5, 6, 7, 3]), tensor([8, 9])]

--- 결과 ---
원문: 나는 학교에 간다
Tensor: tensor([2, 4, 3])

원문: 나는 집에도 가고 학교에도 간다
Tensor: tensor([2, 5, 6, 7, 3])

원문: 학교는 즐겁다
Tensor: tensor([8, 9])



### 2. 패딩(Padding)


In [3]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

sentences = ["나는 학교에 간다", "나는 집에도 가고 학교에도 간다", "학교는 즐겁다"]
tokenized_sentences = [sent.split() for sent in sentences]
print("tokenized : ", tokenized_sentences)

tokens_all = [token for sent in tokenized_sentences for token in sent]
print()
print("tokens_all : ", tokens_all)

vocab_counts = Counter(tokens_all)
print()
print("vocab_counts : ", vocab_counts)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}

for word, _ in vocab_counts.most_common():
    print()
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)
        print("word_to_idx : ", word_to_idx[word])


def encoded(sentence_tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in sentence_tokens]


encoded_list = [encoded(sent, word_to_idx) for sent in tokenized_sentences]
print()
print("encoded_list : ", encoded_list)

tensor_list = [torch.tensor(seq, dtype=torch.long) for seq in encoded_list]
print()
print("tensor_list : ", tensor_list)

# pad_sequence 함수 적용
# batch_first=True : (배치 크기, 최대 문장 길이) 형태로 출력
# padding_value   : 패딩으로 채울 값 (여기서는 <PAD>의 인덱스인 0)
padded_tensors = pad_sequence(
    tensor_list, batch_first=True, padding_value=word_to_idx["<PAD>"]
)
print()
print("padded_tensors : ", padded_tensors)

print()
print("--- 결과 (패딩 적용) ---")
print("최종 Tensor Shape:", padded_tensors.shape)
print(padded_tensors)

tokenized :  [['나는', '학교에', '간다'], ['나는', '집에도', '가고', '학교에도', '간다'], ['학교는', '즐겁다']]

tokens_all :  ['나는', '학교에', '간다', '나는', '집에도', '가고', '학교에도', '간다', '학교는', '즐겁다']

vocab_counts :  Counter({'나는': 2, '간다': 2, '학교에': 1, '집에도': 1, '가고': 1, '학교에도': 1, '학교는': 1, '즐겁다': 1})

word_to_idx :  2

word_to_idx :  3

word_to_idx :  4

word_to_idx :  5

word_to_idx :  6

word_to_idx :  7

word_to_idx :  8

word_to_idx :  9

encoded_list :  [[2, 4, 3], [2, 5, 6, 7, 3], [8, 9]]

tensor_list :  [tensor([2, 4, 3]), tensor([2, 5, 6, 7, 3]), tensor([8, 9])]

padded_tensors :  tensor([[2, 4, 3, 0, 0],
        [2, 5, 6, 7, 3],
        [8, 9, 0, 0, 0]])

--- 결과 (패딩 적용) ---
최종 Tensor Shape: torch.Size([3, 5])
tensor([[2, 4, 3, 0, 0],
        [2, 5, 6, 7, 3],
        [8, 9, 0, 0, 0]])


In [4]:
import torch
from collections import Counter
# from torch.nn.utils.rnn import pad_sequence

# 1~5 단계는 기존 코드와 동일
sentences = ["나는 학교에 간다", "나는 집에도 가고 학교에도 간다", "학교는 즐겁다"]

tokenized_sentences = [sent.split() for sent in sentences]
print("tokenized_sentences : ", tokenized_sentences)

tokens_all = [token for sent in tokenized_sentences for token in sent]
print()
print("tokens_all : ", tokens_all)

vocab_counts = Counter(tokens_all)
print()
print("vocab_counts : ", vocab_counts)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print()
print("word_to_idx : ", word_to_idx)


def encode(sentence_token, vocab):
    return [
        vocab.get(token, word_to_idx["<UNK>"])
        for sent in sentence_token
        for token in sent
    ]


encoded_list = [encoded(sent, word_to_idx) for sent in tokenized_sentences]
print()
print("encoded_list : ", encoded_list)

max_len = max(len(seq) for seq in encoded_list)
print()
print("max_len : ", max_len)

pad_idx = word_to_idx["<PAD>"]

padded_encoded_list = []
for seq in encoded_list:
    padded_seq = seq + [pad_idx] * (max_len - len(seq))
    padded_encoded_list.append(padded_seq)
    print()
    print("padded_seq : ", padded_seq)
    print("padded_encoded_list : ", padded_encoded_list)

padded_tensors = torch.tensor(padded_encoded_list, dtype=torch.long)

for sent, tensor in zip(sentences, padded_tensors):
    print()
    print(f"원문: {sent}")
    print(f"Padded Tensor: {tensor.tolist()}\n")

tokenized_sentences :  [['나는', '학교에', '간다'], ['나는', '집에도', '가고', '학교에도', '간다'], ['학교는', '즐겁다']]

tokens_all :  ['나는', '학교에', '간다', '나는', '집에도', '가고', '학교에도', '간다', '학교는', '즐겁다']

vocab_counts :  Counter({'나는': 2, '간다': 2, '학교에': 1, '집에도': 1, '가고': 1, '학교에도': 1, '학교는': 1, '즐겁다': 1})

word_to_idx :  {'<PAD>': 0, '<UNK>': 1, '나는': 2, '간다': 3, '학교에': 4, '집에도': 5, '가고': 6, '학교에도': 7, '학교는': 8, '즐겁다': 9}

encoded_list :  [[2, 4, 3], [2, 5, 6, 7, 3], [8, 9]]

max_len :  5

padded_seq :  [2, 4, 3, 0, 0]
padded_encoded_list :  [[2, 4, 3, 0, 0]]

padded_seq :  [2, 5, 6, 7, 3]
padded_encoded_list :  [[2, 4, 3, 0, 0], [2, 5, 6, 7, 3]]

padded_seq :  [8, 9, 0, 0, 0]
padded_encoded_list :  [[2, 4, 3, 0, 0], [2, 5, 6, 7, 3], [8, 9, 0, 0, 0]]

원문: 나는 학교에 간다
Padded Tensor: [2, 4, 3, 0, 0]


원문: 나는 집에도 가고 학교에도 간다
Padded Tensor: [2, 5, 6, 7, 3]


원문: 학교는 즐겁다
Padded Tensor: [8, 9, 0, 0, 0]



### 3. 원 - 핫 인코딩(One - Hot Encoding)


In [5]:
import torch
import torch.nn.functional as F

targets = torch.tensor([0, 1, 3])
print("targets : ", targets)

vocab_size = 4

one_hot_vectors = F.one_hot(targets, num_classes=vocab_size)
print()
print("one_hot_vectors : ", one_hot_vectors)

targets :  tensor([0, 1, 3])

one_hot_vectors :  tensor([[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]])


In [6]:
import torch
import torch.nn.functional as F
from collections import Counter

# 1. 실제 입력 데이터 (문장들)
raw_sentences = [
    "나는 사과를 좋아한다",
    "나는 바나나를 좋아한다",
    "나는 사과와 바나나를 먹는다",
]

tokenized_sentences = [sent.split() for sent in raw_sentences]
print("tokenized_sentences : ", tokenized_sentences)

tokens_all = [token for sent in tokenized_sentences for token in sent]
print()
print("tokens_all : ", tokens_all)

vocab_counts = Counter(tokens_all)
print()
print("vocab_counts : ", vocab_counts)

word_to_idx = {}
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print()
print("word_to_idx : ", word_to_idx)

vocab_size = len(word_to_idx)
print()
print("vocab_size : ", vocab_size)

integer_encoded_sentences = []
for sent in tokenized_sentences:
    int_seq = [word_to_idx[token] for token in sent]
    integer_encoded_sentences.append(int_seq)
    print()
    print("int_seq : ", int_seq)
    print("integer_encoded_sentences : ", integer_encoded_sentences)

for i, (origin_sent, int_seq) in enumerate(
    zip(raw_sentences, integer_encoded_sentences)
):
    tensor_seq = torch.tensor(int_seq, dtype=torch.long)
    one_hot_tensor = F.one_hot(tensor_seq, num_classes=vocab_size)

    print()
    print("tensor_seq : ", tensor_seq)
    print("one_hot_tensor : ", one_hot_tensor)

tokenized_sentences :  [['나는', '사과를', '좋아한다'], ['나는', '바나나를', '좋아한다'], ['나는', '사과와', '바나나를', '먹는다']]

tokens_all :  ['나는', '사과를', '좋아한다', '나는', '바나나를', '좋아한다', '나는', '사과와', '바나나를', '먹는다']

vocab_counts :  Counter({'나는': 3, '좋아한다': 2, '바나나를': 2, '사과를': 1, '사과와': 1, '먹는다': 1})

word_to_idx :  {'나는': 0, '좋아한다': 1, '바나나를': 2, '사과를': 3, '사과와': 4, '먹는다': 5}

vocab_size :  6

int_seq :  [0, 3, 1]
integer_encoded_sentences :  [[0, 3, 1]]

int_seq :  [0, 2, 1]
integer_encoded_sentences :  [[0, 3, 1], [0, 2, 1]]

int_seq :  [0, 4, 2, 5]
integer_encoded_sentences :  [[0, 3, 1], [0, 2, 1], [0, 4, 2, 5]]

tensor_seq :  tensor([0, 3, 1])
one_hot_tensor :  tensor([[1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 1, 0, 0, 0, 0]])

tensor_seq :  tensor([0, 2, 1])
one_hot_tensor :  tensor([[1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0]])

tensor_seq :  tensor([0, 4, 2, 5])
one_hot_tensor :  tensor([[1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 1, 0

In [7]:
import torch
import torch.nn.functional as F
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

raw_sentences = [
    "나는 사과를 좋아한다",  # 3개 단어
    "나는 바나나를 좋아한다",  # 3개 단어
    "나는 사과와 바나나를 먹는다",  # 4개 단어
]

tokenized_sentences = [sent.split() for sent in raw_sentences]
print("tokenized_sentences : ", tokenized_sentences)

tokens_all = [token for sent in tokenized_sentences for token in sent]
print()
print("tokens_all : ", tokens_all)

vocab_counts = Counter(tokens_all)
print()
print("vocab_counts : ", vocab_counts)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print()
print("word_to_idx : ", word_to_idx)

vocab_size = len(word_to_idx)
print()
print("vocab_size : ", vocab_size)

tensor_list = [
    torch.tensor(
        [word_to_idx.get(token, word_to_idx["<UNK>"]) for token in sent],
        dtype=torch.long,
    )
    for sent in tokenized_sentences
]

print()
print("tensor_list : ", tensor_list)

all_tensor_seq = pad_sequence(tensor_list, batch_first=True, padding_value=0)
print()
print("all_tensor_seq : ", all_tensor_seq)

one_hot_tensor = F.one_hot(all_tensor_seq, num_classes=vocab_size)
print()
print("one_hot_tensor : ", one_hot_tensor)

print()
print("one_hot_tensor - shape : ", one_hot_tensor.shape)

tokenized_sentences :  [['나는', '사과를', '좋아한다'], ['나는', '바나나를', '좋아한다'], ['나는', '사과와', '바나나를', '먹는다']]

tokens_all :  ['나는', '사과를', '좋아한다', '나는', '바나나를', '좋아한다', '나는', '사과와', '바나나를', '먹는다']

vocab_counts :  Counter({'나는': 3, '좋아한다': 2, '바나나를': 2, '사과를': 1, '사과와': 1, '먹는다': 1})

word_to_idx :  {'<PAD>': 0, '<UNK>': 1, '나는': 2, '좋아한다': 3, '바나나를': 4, '사과를': 5, '사과와': 6, '먹는다': 7}

vocab_size :  8

tensor_list :  [tensor([2, 5, 3]), tensor([2, 4, 3]), tensor([2, 6, 4, 7])]

all_tensor_seq :  tensor([[2, 5, 3, 0],
        [2, 4, 3, 0],
        [2, 6, 4, 7]])

one_hot_tensor :  tensor([[[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 0],
         [0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]]

In [8]:
new_raw_sentences = ["너는 복숭아를 좋아한다", "나는 사과를 아주 잘 먹는다"]

new_tokenized_sentences = [sent.split() for sent in new_raw_sentences]

print("new_tokenized_sentences : ", new_tokenized_sentences)

new_tensor_list = [
    torch.tensor(
        [word_to_idx.get(token, word_to_idx["<UNK>"]) for token in sent],
        dtype=torch.long,
    )
    for sent in new_tokenized_sentences
]

print("new_tensor_list : ", new_tensor_list)

new_all_tensor_seq = pad_sequence(new_tensor_list, batch_first=True, padding_value=0)
print()
print("new_all_tensor_seq : ", new_all_tensor_seq)

new_one_hot_tensor = F.one_hot(new_all_tensor_seq, num_classes=vocab_size)
print()
print("new_one_hot_tensor : ", new_one_hot_tensor)

new_tokenized_sentences :  [['너는', '복숭아를', '좋아한다'], ['나는', '사과를', '아주', '잘', '먹는다']]
new_tensor_list :  [tensor([1, 1, 3]), tensor([2, 5, 1, 1, 7])]

new_all_tensor_seq :  tensor([[1, 1, 3, 0, 0],
        [2, 5, 1, 1, 7]])

new_one_hot_tensor :  tensor([[[0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]]])


### 4. Bag-of-Words(BoW)


In [51]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "나는 사과 를 좋아하 고 사과 를 먹는다",
    "나는 바나나를 좋아한다",
    "나는 사과와 바나나를 모두 먹는다",
]

# vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(corpus)

print("vectorizer.vocabulary")
print(vectorizer.vocabulary_)

print()
print("bow vectors (Dense Array)")
print(bow_matrix.toarray())
# CountVectorizer를 문장을 단어 기준으로 토큰화 시키고 토큰 횟수를 센다
# bow_matrix에 corpus를 CountVectorizer를 적용
# vectorizer.vocabulart_는 CountVectorizer가 토큰화한 결과를 바탕으로 만든 토큰 → 인덱스 딕셔너리
# bow_matrix.toarray()는 CountVectorizer가 기본적으로
# Sparse Matrix(0을 제외한 실제 값의 위치 정보만 가지고 있음)를 반환하기에
# Dense Matrix 즉 일반적인 Array로 변환

vectorizer.vocabulary
{'나는': 0, '사과': 4, '좋아하': 6, '먹는다': 1, '바나나를': 3, '좋아한다': 7, '사과와': 5, '모두': 2}

bow vectors (Dense Array)
[[1 1 0 0 2 0 1 0]
 [1 0 0 1 0 0 0 1]
 [1 1 1 1 0 1 0 0]]


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "I love natural language processing",
    "Language processing is fun and language is powerful",
    "I love machine learning",
]

vect = CountVectorizer()
vect.fit(corpus)

print("단어 사전")
print(vect.get_feature_names_out())

X = vect.transform(corpus)
print()
print("문서 - 단어 카운트 행렬")
print(X.toarray())

단어 사전
['and' 'fun' 'is' 'language' 'learning' 'love' 'machine' 'natural'
 'powerful' 'processing']

문서 - 단어 카운트 행렬
[[0 0 0 1 0 1 0 1 0 1]
 [1 1 2 2 0 0 0 0 1 1]
 [0 0 0 0 1 1 1 0 0 0]]


### 5. TF-IDF(Term Frequency-Inverse Document Frequency)


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "나는 사과를 좋아하고 사과를 먹는다",
    "나는 바나나를 좋아한다",
    "나는 사과와 바나나를 모두 먹는다",
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print("tfidf_matrix")
print(tfidf_matrix)

print()
print("Vocabulary : ", tfidf_vectorizer.vocabulary_)
print()
print("TF-IDF Matrix (Dense Array)")
print(tfidf_matrix.toarray().round(2))

tfidf_matrix
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (3, 8)>
  Coords	Values
  (0, 0)	0.24259369753845733
  (0, 4)	0.8214936700177023
  (0, 6)	0.4107468350088512
  (0, 1)	0.31238355521006117
  (1, 0)	0.4254405389711991
  (1, 3)	0.5478321549274363
  (1, 7)	0.7203334490549893
  (2, 0)	0.3154441510317797
  (2, 1)	0.4061917781433946
  (2, 3)	0.4061917781433946
  (2, 5)	0.5340933749435833
  (2, 2)	0.5340933749435833

Vocabulary :  {'나는': 0, '사과를': 4, '좋아하고': 6, '먹는다': 1, '바나나를': 3, '좋아한다': 7, '사과와': 5, '모두': 2}

TF-IDF Matrix (Dense Array)
[[0.24 0.31 0.   0.   0.82 0.   0.41 0.  ]
 [0.43 0.   0.   0.55 0.   0.   0.   0.72]
 [0.32 0.41 0.53 0.41 0.   0.53 0.   0.  ]]


In [44]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cs

corpus = [
    "나는 사과를 좋아하고 사과를 먹는다",  # 문장 0
    "나는 바나나를 좋아한다",  # 문장 1
    "나는 사과와 바나나를 모두 먹는다",  # 문장 2
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
tfidf_dense = tfidf_matrix.toarray()

print("vectorizer.vocabulary_")
print(vectorizer.vocabulary_)

print()
print("tfidf_dense")
print(tfidf_dense)

tfidf_tensor = torch.tensor(tfidf_dense, dtype=torch.float32)
print()
print("tfidf_tensor")
print(tfidf_tensor)

num_sentences = tfidf_tensor.size(0)
print()
print("num_sentences : ", num_sentences)

pytorch_similarity_matrix = torch.zeros((num_sentences, num_sentences))
print()
print("pytorch_similarity_matrix : ", pytorch_similarity_matrix)

for i in range(num_sentences):
    for j in range(num_sentences):
        sim = F.cosine_similarity(
            tfidf_tensor[i].unsqueeze(0), tfidf_tensor[j].unsqueeze(0)
        )
        pytorch_similarity_matrix[i][j] = sim.item()

        print()
        print("sim")
        print(sim)
        print("pytorch_similarity_matrix")
        print(pytorch_similarity_matrix.numpy().round(4))

cs_matrix = cs(tfidf_matrix)
print()
print("cs_matrix")
print(cs_matrix)

print()
print("cs_matrix.round(4)")
print(cs_matrix.round(4))

print()
print("--- 문장 간 유사도 비교 결과 ---")
print(
    f"문장 0 ('사과를 좋아하고...') vs 문장 1 ('바나나를 좋아한다...'): {pytorch_similarity_matrix[0][1]:.4f}"
)
print(
    f"문장 0 ('사과를 좋아하고...') vs 문장 2 ('사과와 바나나를...'):   {pytorch_similarity_matrix[0][2]:.4f}"
)
print(
    f"문장 1 ('바나나를 좋아한다') vs 문장 2 ('사과와 바나나를...'):   {pytorch_similarity_matrix[1][2]:.4f}"
)

vectorizer.vocabulary_
{'나는': 0, '사과를': 4, '좋아하고': 6, '먹는다': 1, '바나나를': 3, '좋아한다': 7, '사과와': 5, '모두': 2}

tfidf_dense
[[0.2425937  0.31238356 0.         0.         0.82149367 0.
  0.41074684 0.        ]
 [0.42544054 0.         0.         0.54783215 0.         0.
  0.         0.72033345]
 [0.31544415 0.40619178 0.53409337 0.40619178 0.         0.53409337
  0.         0.        ]]

tfidf_tensor
tensor([[0.2426, 0.3124, 0.0000, 0.0000, 0.8215, 0.0000, 0.4107, 0.0000],
        [0.4254, 0.0000, 0.0000, 0.5478, 0.0000, 0.0000, 0.0000, 0.7203],
        [0.3154, 0.4062, 0.5341, 0.4062, 0.0000, 0.5341, 0.0000, 0.0000]])

num_sentences :  3

pytorch_similarity_matrix :  tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

sim
tensor([1.])
pytorch_similarity_matrix
[[1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

sim
tensor([0.1032])
pytorch_similarity_matrix
[[1.     0.1032 0.    ]
 [0.     0.     0.    ]
 [0.     0.     0.    ]]

sim
tensor([0.2034])
pytorch_similarity_matrix
[[1.     0.103

In [ ]:
import torch

# 단일 문장 정수 인코딩: [나는, 사과를, 좋아한다] -> [1, 4, 2]
single_sentence = torch.tensor([1, 4, 2])
print("원본 Shape:", single_sentence.shape)
# 출력: torch.Size([3]) (1차원, 단어 수 3개)
print(single_sentence)

# 0번째 위치에 배치(Batch) 차원 추가
batched_sentence = single_sentence.unsqueeze(0)
print("unsqueeze(0) 후 Shape:", batched_sentence.shape)
# 출력: torch.Size([1, 3]) (2차원, Batch=1, 문장길이=3)
print(batched_sentence)

원본 Shape: torch.Size([3])
tensor([1, 4, 2])
unsqueeze(0) 후 Shape: torch.Size([1, 3])
tensor([[1, 4, 2]])


In [49]:
import torch
import torch.nn as nn

vocab_size = 8

all_tensor_seq = torch.tensor(
    [[1, 4, 2, 0], [1, 3, 2, 0], [1, 6, 3, 5]], dtype=torch.long
)
print("all_tensor_seq")
print(all_tensor_seq)
embedding_dim = 4

embedding_layer = nn.Embedding(
    num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0
)

dense_embedded_tensor = embedding_layer(all_tensor_seq)

print()
print("입력 정수 Tensor Shape")
print(all_tensor_seq.shape)

print()
print("nn.Embedding 출력 Tensor Shape")
print(dense_embedded_tensor[0])
print(dense_embedded_tensor[1])
print(dense_embedded_tensor[2])
print()
print("문장 1의 첫 단어 ('나는', idx=1)의 밀집 벡터")
print(dense_embedded_tensor[0, 0])

print()
print("문장 1의 패딩('<PAD>')의 밀집 벡터")
print(dense_embedded_tensor[0, 3])

all_tensor_seq
tensor([[1, 4, 2, 0],
        [1, 3, 2, 0],
        [1, 6, 3, 5]])

입력 정수 Tensor Shape
torch.Size([3, 4])

nn.Embedding 출력 Tensor Shape
tensor([[ 0.3409,  0.3389, -0.9747, -0.3841],
        [ 1.1374,  0.9703, -1.3566,  0.5103],
        [ 1.0264,  1.8053, -0.7382, -0.9921],
        [ 0.0000,  0.0000,  0.0000,  0.0000]], grad_fn=<SelectBackward0>)
tensor([[ 0.3409,  0.3389, -0.9747, -0.3841],
        [ 1.9766, -0.9083,  0.9151,  0.0930],
        [ 1.0264,  1.8053, -0.7382, -0.9921],
        [ 0.0000,  0.0000,  0.0000,  0.0000]], grad_fn=<SelectBackward0>)
tensor([[ 0.3409,  0.3389, -0.9747, -0.3841],
        [-1.1109, -1.1204,  0.5569, -0.5083],
        [ 1.9766, -0.9083,  0.9151,  0.0930],
        [ 0.7088,  0.8117,  0.3810,  1.2935]], grad_fn=<SelectBackward0>)

문장 1의 첫 단어 ('나는', idx=1)의 밀집 벡터
tensor([ 0.3409,  0.3389, -0.9747, -0.3841], grad_fn=<SelectBackward0>)

문장 1의 패딩('<PAD>')의 밀집 벡터
tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)
